In [ ]:
using Pkg
Pkg.activate("..")
using MajoranaPropagation
using Revise
using PauliPropagation
using Plots
using LaTeXStrings
using Measures
using PythonCall

  Activating project at `c:\Users\jugro\Desktop\8Sem_ETH\Research Project\Simulations\path\to\project`


In [2]:
#important functions
function expand_tensor(pairs)
    result = [Dict{Int,Any}()]
    for pair in pairs
        new_result = []
        for r in result
            for p in pair
                merged = copy(r)
                merge!(merged, p)
                push!(new_result, merged)
            end
        end
        result = new_result
    end

    return result
end

function to_fock(state, N)
    ups = Int[]
    dns = Int[]

    for (site, spin) in state

        if spin == :updn || spin == :dnup
            push!(ups, site)
            push!(dns, site)

        elseif spin == :up
            push!(ups, site)

        elseif spin == :dn
            push!(dns, site)
        end
    end

    return FockState(N, sort(ups), sort(dns))
end;

In [23]:
#System
is_spinful = true
N_x = 4
N_y = 7
N_spinful_sites = N_x * N_y
site_doublon = 19

t = 1.
U = 4. #U=0 also? ->weight const, look a t paper

n_layers = 3
dt = 0.1

min_abs_coeff = 1.e-2
max_unpaired = 4
unpaired_mask = create_unpaired_mask(2 * N_spinful_sites);

topo = rectangletopology(N_x, N_y,periodic=true)

circ_single = []
thetas_single = []

#up hoppings 
for (i, j) in topo
    push!(circ_single, FermionicGate(:hopup, [i, j]))
    push!(thetas_single, -t * dt)
end

#down hoppings 
for (i, j) in topo
    push!(circ_single, FermionicGate(:hopdn, [i, j]))
    push!(thetas_single, -t * dt)
end

#on-site repulsion 
for i = 1:N_spinful_sites
    push!(circ_single, FermionicGate(:nupndn, i))
    push!(thetas_single, U * dt)
end


#construction of initial state from doublon and pairs
site_pairs = [(1,8), (3,4), (5,12), (6,13), (7,14), (9,16), (10,17), (11,18), (15,22), (23,24), (25,26), (20,27), (21,28)]
pairs = [ [Dict(site_doublon => :updn)] ] #doublon

for (s1, s2) in site_pairs
    push!(pairs, [Dict(s1 => :up, s2 => :dn), Dict(s1 => :dn, s2 => :up)])
end

#preparing inital states
states = expand_tensor(pairs)
fock_states = [to_fock(s, N_spinful_sites) for s in states]
initial_state = [fock_states[i] for i in 1:length(fock_states)]
coeffs = [1/(sqrt(2)^13) for i in 1:length(initial_state)] #13 superpositions, each with coeff 1/sqrt(2)

#observables for later use
obs_doubl = MajoranaSum(Float64, N_spinful_sites, is_spinful)
for i=1:N_spinful_sites
    add!(obs_doubl, :nupndn, i)
end
obs_doubl = VectorMajoranaSum(obs_doubl)

@show initial_state

8192-element Vector{FockState{PauliPropagation.UInt112}}:
 Fock state with 28 fermions at positions
    ↑: 1, 3, 5, 6, 7, 9, 10, 11, 15, 19, 20, 21, 23, 25
    ↓: 4, 8, 12, 13, 14, 16, 17, 18, 19, 22, 24, 26, 27, 28

 Fock state with 28 fermions at positions
    ↑: 1, 3, 5, 6, 7, 9, 10, 11, 15, 19, 20, 23, 25, 28
    ↓: 4, 8, 12, 13, 14, 16, 17, 18, 19, 21, 22, 24, 26, 27

 Fock state with 28 fermions at positions
    ↑: 1, 3, 5, 6, 7, 9, 10, 11, 15, 19, 21, 23, 25, 27
    ↓: 4, 8, 12, 13, 14, 16, 17, 18, 19, 20, 22, 24, 26, 28

 Fock state with 28 fermions at positions
    ↑: 1, 3, 5, 6, 7, 9, 10, 11, 15, 19, 23, 25, 27, 28
    ↓: 4, 8, 12, 13, 14, 16, 17, 18, 19, 20, 21, 22, 24, 26

 Fock state with 28 fermions at positions
    ↑: 1, 3, 5, 6, 7, 9, 10, 11, 15, 19, 20, 21, 23, 26
    ↓: 4, 8, 12, 13, 14, 16, 17, 18, 19, 22, 24, 25, 27, 28

 Fock state with 28 fermions at positions
    ↑: 1, 3, 5, 6, 7, 9, 10, 11, 15, 19, 20, 23, 26, 28
    ↓: 4, 8, 12, 13, 14, 16, 17, 18, 19, 21, 22, 

# Testing initial state

We start testing the initial state, by measuring expectation values of simple on-site observables. Since the computation is roughly 1 minute, we look at the sites 1,2,19 and 28. (2: hole, 19: doublon, rest in a superposition). 

In [ ]:
#1
obs_1up = VectorMajoranaSum(MajoranaSum(N_spinful_sites, :nup, 1)) 
obs_1dn = VectorMajoranaSum(MajoranaSum(N_spinful_sites, :ndn, 1))
#2
obs_2up = VectorMajoranaSum(MajoranaSum(N_spinful_sites, :nup, 2))
obs_2dn = VectorMajoranaSum(MajoranaSum(N_spinful_sites, :ndn, 2))
#19
obs_19up = VectorMajoranaSum(MajoranaSum(N_spinful_sites, :nup, site_doublon))
obs_19dn = VectorMajoranaSum(MajoranaSum(N_spinful_sites, :ndn, site_doublon))
#28
obs_28up = VectorMajoranaSum(MajoranaSum(N_spinful_sites, :nup, N_spinful_sites))
obs_28dn = VectorMajoranaSum(MajoranaSum(N_spinful_sites, :ndn, N_spinful_sites))

# Use println to ensure each result is on its own line
#@show overlapwithfock(obs_1up, initial_state, coeffs); #0.5
#@show overlapwithfock(obs_1dn, initial_state, coeffs); #0.5
#@show overlapwithfock(obs_2up, initial_state, coeffs); #0
#@show overlapwithfock(obs_2dn, initial_state, coeffs); #0
#@show overlapwithfock(obs_19up, initial_state, coeffs); #1
#@show overlapwithfock(obs_19dn, initial_state, coeffs); #1
#@show overlapwithfock(obs_28up, initial_state, coeffs); #0.5
#@show overlapwithfock(obs_28dn, initial_state, coeffs); #0.5

#lets try one non-local observable, e.g. doublon number
@show overlapwithfock(obs_doubl, initial_state, coeffs); #1

0.9999999999990905

In [ ]:
#simulation, do not run for large number of layers
res1 = zeros(n_layers+1)
length1 = zeros(n_layers+1)

for j = 1:n_layers 
    print("$j ")
    propagate!(circ_single, obs_doubl, thetas_single; min_abs_coeff=min_abs_coeff)
    res1[j+1] = overlapwithfock(obs_doubl, initial_state, coeffs)
    length1[j+1] = length(obs_doubl)
end

ev_plot = plot(xlabel = "Layer", ylabel = L"<O>"*"at coeff. cutoff " *"10^{-4}", legend = :bottomright, legend_font_pointsize = 14)
plot!(ev_plot, 0:n_layers, res1, label=L"n_{doublon}", marker=:circle)

length_plot = plot(xlabel = "Layer",ylabel = "Number of Majorana strings", yscale = :log10, legend = :bottomright,legend_font_pointsize = 12)
plot!(length_plot, 0:n_layers, length1, label=L"n_{doublon}", marker=:circle)

plot(ev_plot, length_plot, layout=(1,2), size=(1000,400),left_margin = 10mm, bottom_margin = 5mm, right_margin = 5mm)